# 07 · Similarity search — the payoff

Each shot becomes a **77-dimensional vector** (core physics + per-beam NBI + per-gyrotron ECH + per-valve gas flow). Features are min-max normalized across the database and compared with **cosine similarity**.

Two query modes:
1. **By shot number** — "show me the 5 shots most like #144229."
2. **By desired parameters** — a physicist enters the planned setup (e.g. beams 30L/30R at 2 MW, I<sub>p</sub> = 1 MA, κ = 1.8); unspecified parameters default to the database mean.

Results are enriched with logbook sentiment and objective success so the operator can see which similar shots actually worked.

**Sanity check:** the nearest neighbor of shot 144229 is 144230 — the very next shot in the same session.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("/path/to/outputs/shot_parameters_v3.csv")
df_sentiment = pd.read_csv("/path/to/outputs/shot_sentiment.csv")
df_success = pd.read_csv("/path/to/outputs/shot_success_scores.csv")

print("Parameter database:", len(df), "shots")
print("Sentiment-scored shots:", len(df_sentiment))
print("Success-scored shots:", len(df_success))
print("Columns available:", len(df.columns))

In [ ]:
NBI_BEAMS = ["NB15L", "NB15R", "NB21L", "NB21R", "NB30L", "NB30R", "NB33L", "NB33R"]

for beam in NBI_BEAMS:
    col = f"peak_pabs_{beam}"
    non_null = df[col].notna().sum()
    print(f"{beam}: {non_null} / {len(df)} shots ({100*non_null/len(df):.1f}%)")

In [ ]:
core_columns = [
    "peak_temp", "mean_temp",
    "peak_density", "mean_density",
    "peak_ip", "mean_ip",
    "kappa", "tritop", "tribot", "betan", "q95",
    "peak_ts_tangential_density", "mean_ts_tangential_density",
    "peak_ts_tangential_temp", "mean_ts_tangential_temp",
]

nbi_beam_columns = [f"{stat}_pabs_{beam}" for beam in NBI_BEAMS for stat in ("peak", "mean")]

ech_towers = ["BORIS", "CHEWBACCA", "DOROTHY", "HAN", "KATYA", "LEIA",
              "LION", "LUKE", "NASA", "NATASHA", "R2D2", "SCARECROW"]
echpwr_columns = [f"{stat}_echpwr_{tower}" for tower in ech_towers for stat in ("peak", "mean")]

gas_valves = ["GASA", "GASB", "GASC", "GASD", "GASE", "LOB1", "LOB2", "PFX1", "PFX2", "PFX3", "UOB"]
gasflow_columns = [f"{stat}_gasflow_{valve}" for valve in gas_valves for stat in ("peak", "mean")]

param_columns = core_columns + nbi_beam_columns + echpwr_columns + gasflow_columns

df_params = df[["shot_number"] + param_columns].copy()
df_params[param_columns] = df_params[param_columns].fillna(0)

scaler = MinMaxScaler()
df_normalized = df_params.copy()
df_normalized[param_columns] = scaler.fit_transform(df_params[param_columns])

print("Normalization done. Shape:", df_normalized.shape)
print("Total parameters used:", len(param_columns))

In [ ]:
def enrich_shot(shot_num, df_sentiment, df_success):
    entry = {}

    sentiment_row = df_sentiment[df_sentiment["shot_number"] == shot_num]
    if len(sentiment_row) > 0:
        sentiment_row = sentiment_row.iloc[0]
        entry["sentiment"] = sentiment_row["sentiment"]
        entry["sentiment_source"] = sentiment_row["source"]
    else:
        entry["sentiment"] = None
        entry["sentiment_source"] = None

    success_row = df_success[df_success["shot_number"] == shot_num]
    if len(success_row) > 0:
        success_row = success_row.iloc[0]
        entry["overall_success"] = success_row["overall_success"]
        entry["ip_percent_diff"] = success_row["ip_percent_diff"]
        entry["nbi_percent_diff"] = success_row["nbi_percent_diff"]
    else:
        entry["overall_success"] = None
        entry["ip_percent_diff"] = None
        entry["nbi_percent_diff"] = None

    return entry

print("Enrichment helper defined")

In [ ]:
def find_similar_shots(query_shot_number, df_normalized, df_sentiment, df_success, n=5):
    query_row = df_normalized[df_normalized["shot_number"] == query_shot_number]
    if len(query_row) == 0:
        print("Shot", query_shot_number, "not found in database.")
        return None

    query_vector = query_row[param_columns].values
    all_vectors = df_normalized[param_columns].values
    all_shot_numbers = df_normalized["shot_number"].values

    similarities = cosine_similarity(query_vector, all_vectors)[0]
    sorted_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in sorted_indices:
        shot_num = all_shot_numbers[idx]
        if shot_num == query_shot_number:
            continue
        entry = {"shot_number": int(shot_num), "similarity": round(float(similarities[idx]), 4)}
        entry.update(enrich_shot(shot_num, df_sentiment, df_success))
        results.append(entry)
        if len(results) == n:
            break

    return results

print("Search-by-shot function defined")

In [ ]:
def find_shots_by_specs(desired_specs, df_normalized, df_params, df_sentiment, df_success, n=5):
    query_values = {col: df_params[col].mean() for col in param_columns}

    for key, value in desired_specs.items():
        if key in param_columns:
            query_values[key] = value
        else:
            print("Warning: unknown parameter", key)

    query_df = pd.DataFrame([query_values], columns=param_columns)
    query_normalized = scaler.transform(query_df)

    all_vectors = df_normalized[param_columns].values
    all_shot_numbers = df_normalized["shot_number"].values

    similarities = cosine_similarity(query_normalized, all_vectors)[0]
    sorted_indices = np.argsort(similarities)[::-1]

    results = []
    for idx in sorted_indices:
        shot_num = all_shot_numbers[idx]
        entry = {"shot_number": int(shot_num), "similarity": round(float(similarities[idx]), 4)}
        entry.update(enrich_shot(shot_num, df_sentiment, df_success))
        results.append(entry)
        if len(results) == n:
            break

    return results

print("Search-by-specs function defined")

In [ ]:
results = find_similar_shots(144229, df_normalized, df_sentiment, df_success, n=5)
if results:
    print("Shots most similar to 144229:\n")
    for r in results:
        print(f"Shot: {r['shot_number']} | Similarity: {r['similarity']} | "
              f"Sentiment: {r['sentiment']} | Success: {r['overall_success']}")

In [ ]:
desired = {
    "peak_pabs_NB30L": 2e6,
    "peak_pabs_NB30R": 2e6,
    "peak_ip": 1e6,
    "kappa": 1.8
}

results = find_shots_by_specs(desired, df_normalized, df_params, df_sentiment, df_success, n=5)
print("Shots matching desired specs:\n")
for r in results:
    print(f"Shot: {r['shot_number']} | Similarity: {r['similarity']}")

top = df_params[df_params["shot_number"] == results[0]["shot_number"]].iloc[0]
print("\nTop match actual per-beam values:")
for beam in NBI_BEAMS:
    print(f"  {beam}: peak={top[f'peak_pabs_{beam}']}")